# Bootstrap

## Outline

- 简介
- 原理：总体、样本、经验样本 (自助样本)、

## 备用资料

- https://www.stats.ox.ac.uk/~caron/teaching/sb1b/lecturebootstrap.pdf
  - 对基本概念的介绍部分写的不错  

- A better way to bootstrap pairs [-Link-](https://core.ac.uk/download/pdf/7308338.pdf)


- CHUNG-MING KUAN, LECTURE ON BOOTSTRAP, [-PDF-](https://homepage.ntu.edu.tw/~ckuan/pdf/Lec-Boot_0905.pdf)



# B361：翻译-3页：手动实现排序检验和Bootstrap

Lars Ängquist, 2010, Stata Tip 92: Manual Implementation of Permutations and Bootstraps, Stata Journal, 10(4): 686–688. [-PDF-](https://journals.sagepub.com/doi/pdf/10.1177/1536867X1101000410)

可以做适当扩展，比如实现 cluster Bootstrap


参考资料

- Cameron, A., P. Trivedi, 2009, “Microeconometrics Using Stata”, Stata Press.
- Chernick, M. R., 2008, “Bootstrap methods: A guide for practitioners and researchers”, Wiley-Interscience.
- Davidson, R., J. MacKinnon, 2006, “The power of bootstrap and asymptotic tests”, Journal of Econometrics, 133 (2), 421-441.
- Good, P., 2005, “Permutation, parametric, and bootstrap tests of hypotheses”, Springer, New York.
- Hardle, W., J. Horowitz, J. Kreiss, 2003, “Bootstrap methods for time series”, International Statistical Review, 71 (2), 435-460.
- MacKinnon, J., 2002, “Bootstrap inference in econometrics”, Canadian Journal of Economics, 35 (4), 615-645.
- MacKinnon, J., 2006, “Bootstrap methods in econometrics”, Economic Record, 82 (S1), S2-S18.
- Ruiz, E., L. Pascual, 2002, “Bootstrapping Financial Time Series”, Journal of Economic Surveys, 16 (3), 271-300.
- Wehrens, R., H. Putter, L. Buydens, 2000, “The bootstrap: a tutorial”, Chemometrics and Intelligent Laboratory Systems, 54 (1), 35-52.


## 参考文献

- Flachaire, E. (1999). A better way to bootstrap pairs. Working paper. [-PDF-](https://core.ac.uk/download/pdf/7308338.pdf)


--- - --
<!-- backgroundColor: #C1CDCD -->

## 系数的标准误 (B)：

### 基于再抽样的推断方法

- Jackknife 
- Bootstrap
--- - --
<!-- backgroundColor: #FFFFF9 -->
### 系数的标准误: 基本思想

> Source: [Stewart - Princeton, P.23](https://scholar.princeton.edu/sites/default/files/bstewart/files/lecture5_handout2020.pdf)

- OLS 是一个估计量：我们随机抽取样本 &rarr; 获得参数估计
- 这个过程与计算样本的均值、标准差等统计量的过程并无本质差别
- 因此，$\small \widehat{\beta}$ 具有其抽样分布特征 &rarr; sampling variance/standard error, etc.

<br>

  ![w:800](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/OLS_Smapling-distribution-beta_01.png)


--- - --
<!-- backgroundColor: #e6e6fa -->

### 各种方法：如何计算/获取系数估计值的标准误？

* **M1**: 随机抽样 
* **M2**: 精确推断：依赖于特定的分布假设，如 $\varepsilon \sim N(\cdot)$
  - Homo: $\quad \varepsilon_i \sim N(0, \sigma^2)$
    - `. reg y x`
  - Het &rarr; White (1980) $\quad \varepsilon_i \sim N(0, \sigma_i^2)$ 
    - `. reg y x, robust`
  - Clustered adj SE: $\quad \varepsilon_i \sim N(0, \sigma_j^2), j=1,2\cdots J,\ Corr(\varepsilon_{ji},\varepsilon_{js}) \neq 0$
    - `. reg y x, vce(cluster industry)` 
* **M3**: 再抽样 (re-sampling)
  - Jackknife (去一法, Leave one out, LOO) &rarr; `reg y x, vce(jackknife)`
  - Bootstrap (自举法/自抽样，BS) &rarr; `reg y x, vce(bs, reps(500))`

--- - --
<!-- backgroundColor: #FFFFF9 -->
### DGP
$y_i= 1 + 2x_i + u$
$x\sim N(0,0.2^2),\  u\sim N(0,1)$

```stata
set scheme white_tableau 
*-DGP
clear 
set seed 13579
set obs 50
gen u = rnormal(0,2)
gen x = rnormal(0,0.2)
gen y = 1 + 2*x + u 
save "$D/simdata_ols.dta", replace 
```

> **Goal:** 用这份模拟数据来分析 M1-M3 各类方法下产生的标准误有何差异

--- - --

### **M1**: 随机抽样 
$\small y_i= 1 + 2x_i + u$
$\small x\sim N(0,0.2^2),\  u\sim N(0,1)$

![bg left:42% w:560](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Fig_OLS__smaple_SE_01.png)

```stata
*-OLS 系数标准误：随机抽样
use "$D/simdata_ols.dta", clear 
tw scatter y x 
forvalues j=1/30{
  preserve 
    sample 70 // 随机抽取 #% 的观察值
    addplot: lfit y x, ///
             lc(gray%30) legend(off)
  restore 
}
addplot: lfit y x, lw(*2) lc(orange)
```
--- - --

### **M1**: 随机抽样：如何计算 $se(\widehat{\beta})$ ?
1. Sampling: $\small s_1$ &rarr; $\small \widehat{\beta}_1^{s}$; $\small s_2$ &rarr; $\small\widehat{\beta}_2^{s}$ ...
2. $\small\widehat{\beta}_k^{s} \in \{\widehat{\beta}_1^{s}, \widehat{\beta}_2^{s}, \cdots, \widehat{\beta}_K^{s}\}$
3. $\small se(\widehat{\beta}) = s.d(\widehat{\beta}_k^{s})$ 

![bg left:42% w:560](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Fig_OLS__smaple_SE_01.png)

```stata
use "$D/simdata_ols.dta", clear 
gen bk = .  // 用于存储系数估计值
forvalues j=1/30{
  preserve 
    qui sample 70 // 随机抽取 #% 的观察值
    qui reg y x 
  restore 
  qui replace bk = _b[x] in `j'
}

. sum bk 
  Variable |  Obs        Mean    Std. dev.       Min        Max
-----------+---------------------------------------------------
        bk |   30    3.847519    .9627126   2.297643   7.328576

. reg y x , noheader
----------------------------------------------------------------
 y | Coefficient  Std. err.    t    P>|t|   [95% conf. interval]
---+------------------------------------------------------------
 x |      3.812      1.376   2.77   0.008      1.047       6.578
----------------------------------------------------------------
```

--- - --

### **M2**: 精确推断

<br>
<br>

![bg left w:660](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Fig_OLS_SE_normal_01.png)



```stata
use "$D/simdata_ols.dta", clear 

tw scatter y x || lfitci y x
```

--- - --

### **M3**: 二次抽样 (a) - Jackknife 

![bg left w:660](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Fig_OLS_SE_jk_01.png)

```stata
use "$D/simdata_ols.dta", clear 

tw scatter y x 
local N = _N

forvalues j=1/`N'{
  preserve 
    drop in `j' //每次扔掉一个观察值
    addplot: lfit y x, ///
             lc(gray%30) legend(off)
  restore 
}
addplot: lfit y x, lw(*2) lc(orange)
```

--- - --

### **M3b**: 二次抽样 (b) - Bootstrap

![bg left w:660](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Fig_OLS_SE_bs_01.png)

```stata
use "$D/simdata_ols.dta", clear 

tw scatter y x 
local N = _N
forvalues j=1/50{  // BS 50 次
  preserve 
    bsample // Bootstrap sampling
    addplot: lfit y x, ///
             lc(gray%30) legend(off)
  restore 
}
addplot: lfit   y x, lw(*2) lc(orange)
addplot: lfitci y x, fcolor(yellow%20) nofit
```
- Note: 黄色阴影是基于精确推断 (Normal) 得到的 $95\%$ 置信区间
